# Initial configuration

In [ ]:
import os
import sys
import torch
from pathlib import Path
from matplotlib import font_manager
import matplotlib.pyplot as plt
import random
import numpy as np
from modules.utils import set_seed, get_device
from modules.config import *

# Setting seed
set_seed()

# Device configuration
device = get_device()
print(f"Using device: {device}")

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Paths configuration loaded")
print(f"   • Dataset: {DATASET_PATH}")
print(f"   • Output: {OUTPUT_DIR}")
print(f"   • Model: {MODEL_SAVE_PATH}")

# Dataset and model setup

In [ ]:
import os
import torch
from torch.utils.data import Subset
from torchinfo import summary
from modules.dataset import ETL9Dataset
from modules.models import build_multi_head_model
import modules.transforms as transforms
from modules.utils import get_device
from modules.data_loaders import create_splits, get_dataloaders

print(f"Indexing full dataset (Limit: {MAX_CLASSES_LIMIT})...")

# Device configuration
print(f"Using device: {device}")

# Data Augmentation and Transforms
train_transforms, val_transforms = transforms.get(IMG_SIZE, CHANEL_SIZE)

# Load and Index Dataset
if not os.path.exists(DATASET_PATH):
    raise FileNotFoundError(f"Root directory not found at: {DATASET_PATH}")

print("Indexing full dataset...")
full_dataset = ETL9Dataset(root_dir=DATASET_PATH, transform=None, max_classes=MAX_CLASSES_LIMIT, json_path=COMPONENTS_JSON_PATH)

if len(full_dataset) == 0:
    raise ValueError("Dataset is empty. Check path and unpacked folders.")


# Getting indexes for data splitting
train_idx, val_idx, test_idx = create_splits(len(full_dataset), 0.7, 0.2)

print("Preparing subsets...")
train_dataset_augmented = ETL9Dataset(root_dir=DATASET_PATH, transform=train_transforms, max_classes=MAX_CLASSES_LIMIT, json_path=COMPONENTS_JSON_PATH) # Should be here after data split
base_dataset = ETL9Dataset(root_dir=DATASET_PATH, transform=val_transforms, max_classes=MAX_CLASSES_LIMIT, json_path=COMPONENTS_JSON_PATH) # Should be here after data split

train_subset = Subset(train_dataset_augmented, train_idx)
val_subset = Subset(base_dataset, val_idx)
test_subset = Subset(base_dataset, test_idx)



model = build_multi_head_model(
    num_classes=len(full_dataset.classes), 
    components_classes=len(full_dataset.comp_to_idx)
)
print(f"{type(model).__name__} configured for {len(full_dataset.classes)} classes.")
print(" === Model Architecure Detailed ===")
print(summary(model, input_size=(BATCH_SIZE, CHANEL_SIZE, IMG_SIZE, IMG_SIZE))) # Printing explicit information about the architecture

# Dataset distribution

In [ ]:
from collections import Counter
class_counts = Counter([sample[1] for sample in full_dataset.samples])
unique_counts = set(class_counts.values())
print(f"All classes have same number of images: {len(unique_counts) == 1}")
print(f"Range: min={min(unique_counts)}, max={max(unique_counts)}")

# Optuna hyperparameter optimization

In [ ]:
from functools import partial
from modules.optuna import objective
from modules.models import build_multi_head_model
from modules.data_loaders import get_dataloaders_reduced
import optuna

if OPTUNA_ENABLED:
    study = optuna.create_study(direction="maximize", pruner=optuna.pruners.MedianPruner())
    get_dataloaders_fn = partial(
        get_dataloaders_reduced,
        full_dataset=full_dataset,
        train_dataset_augmented=train_dataset_augmented,
    )
    
    study.optimize(
        lambda trial: objective(
            trial, 
            get_dataloaders_fn=get_dataloaders_fn, 
            build_model_fn=build_multi_head_model, 
            device=device, 
            optuna_epochs=OPTUNA_EPOCHS
        ), 
        n_trials=OPTUNA_TRIALS
    )
    
    optuna.visualization.plot_optimization_history(study).show()
    optuna.visualization.plot_param_importances(study).show()
    optuna.visualization.plot_parallel_coordinate(study).show()
    
    print("\n--- Optimization finished ---")
    print(f"Best Accuracy: {study.best_value:.4f}")
    print(f"Best parameters: {study.best_params}")

# Model training

In [ ]:
import json
from modules.train_utils import EarlyStopping
from modules.train_model import train_model
from modules.train_model import train_kaggle
from modules.train_utils import setup_training_tools
from modules.models import build_multi_head_model
from modules.data_loaders import create_splits, get_dataloaders

if OPTUNA_ENABLED:
    # Execution with Optuna computed parameters
    # Getting best parameters
    best_params = study.best_params
    print(f"Best params: {best_params}")
else:
    best_params = {
        'batch_size': BATCH_SIZE,
        'lr': LEARNING_RATE,
        'weight_decay': WEIGHT_DECAY
    }

# Getting models and dataloaders
train_loader, val_loader, test_loader = get_dataloaders(train_subset, val_subset, test_subset, batch_size=best_params['batch_size'])
print("DataLoaders ready")
model = build_multi_head_model(
    num_classes=len(full_dataset.classes),
    components_classes=len(full_dataset.comp_to_idx)
)


optimizer, scheduler, criterion_kanji, criterion_components = setup_training_tools(
    model, 
    best_params['lr'], 
    best_params['weight_decay']
)

early_stopping = EarlyStopping(patience=5)


RESUME_TRAINING = False # Set true to resume training from last checkpoint
if RESUME_TRAINING and os.path.exists(os.path.join(OUTPUT_DIR, 'last_checkpoint.pth')):
    checkpoint = torch.load(os.path.join(OUTPUT_DIR, 'last_checkpoint.pth'))
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint['epoch']
    history = checkpoint['history']
    best_acc = checkpoint['best_acc']
    print(f"Restarting from epoch {start_epoch}")
else:
    start_epoch = 0
    history = None
    best_acc = 0.0

# Training
model, history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion_kanji=criterion_kanji,            
    criterion_components=criterion_components,
    optimizer=optimizer,
    num_epochs=NUM_EPOCHS,
    output_dir=OUTPUT_DIR,
    model_save_path=MODEL_SAVE_PATH,
    device=device,
    early_stopping=early_stopping,
    scheduler=scheduler,
    start_epoch=start_epoch,
    best_acc=best_acc,
    history=history
)

# Save history and classes
with open(HISTORY_SAVE_PATH, 'w') as f:
    json.dump(history, f)
print(f"History saved to: {HISTORY_SAVE_PATH}")

with open(COMPONENTS_SAVE_PATH, 'w') as f:
    json.dump(full_dataset.comp_to_idx, f, ensure_ascii=False)
print(f"Components saved to: {COMPONENTS_SAVE_PATH}")

with open(CLASSES_SAVE_PATH, 'w') as f:
    json.dump(full_dataset.classes, f, ensure_ascii=False)  
print(f"Classes saved to: {CLASSES_SAVE_PATH}")

# Training metrics

In [ ]:
import json
import matplotlib.pyplot as plt

def plot_training_history():
    with open(HISTORY_SAVE_PATH, 'r') as f:
        history = json.load(f)
    
    acc = history['train_acc']
    val_acc = history['val_acc']
    comp_acc = history['train_comp_acc']
    val_comp_acc = history['val_comp_acc']
    loss = history['train_loss']
    val_loss = history['val_loss']
    epochs_range = range(1, len(acc) + 1)
    
    plt.figure(figsize=(18, 5))
    
    # Kanji Accuracy plot
    plt.subplot(1, 3, 1)
    plt.plot(epochs_range, acc, label='Training Accuracy', color='blue')
    plt.plot(epochs_range, val_acc, label='Validation Accuracy', color='orange')
    plt.title('Kanji Classification Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend(loc='lower right')
    plt.grid(True)
    
    # Loss plot
    plt.subplot(1, 3, 2)
    plt.plot(epochs_range, loss, label='Training Loss', color='blue')
    plt.plot(epochs_range, val_loss, label='Validation Loss', color='orange')
    plt.title('Training and Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend(loc='upper right')
    plt.grid(True)
    
    # Components Accuracy plot
    plt.subplot(1, 3, 3)
    plt.plot(epochs_range, comp_acc, label='Training Component Acc', color='green')
    plt.plot(epochs_range, val_comp_acc, label='Validation Component Acc', color='red')
    plt.title('Component Classification Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend(loc='lower right')
    plt.grid(True)
    
    plt.tight_layout()
    plt.show()

plot_training_history()

# Test evaluation

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import font_manager
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import classification_report, top_k_accuracy_score
import torch
from tqdm import tqdm
import json
from modules import fonts
import modules.image_processing as ip

# Loading a displayable japanese font
fonts.load_font(FONT_PATH)

# Function to display some tests
def visualize_errors_and_metrics(model, test_loader, class_names):
    model.eval() # Set model to evaluation mode
    all_preds = []
    all_labels = []
    all_probs = []
    mistakes = [] 
    MISTAKES_TO_DISPLAY = 20
    ROWS = 4
    COLUMNS = 5
    
    print("Analyzing Test data...")
    
    with torch.no_grad():
        for inputs, labels in tqdm(test_loader, desc="Processing"):
            inputs = inputs.to(device)
            l_kanji = labels['kanji'].to(device)
            
            # Forward pass: El modelo devuelve (kanji, components)
            outputs_kanji, _ = model(inputs)
            
            probs = torch.nn.functional.softmax(outputs_kanji, dim=1)
            conf, preds = torch.max(probs, 1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(l_kanji.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            
            wrong_indices = (preds != l_kanji).nonzero()
            for idx in wrong_indices:
                idx = idx.item()
                if len(mistakes) < MISTAKES_TO_DISPLAY: # Just displaying limited mistakes
                    img_tensor = inputs[idx]
                    img = ip.denormalize(img_tensor)
                    
                    mistakes.append({
                        'img': img,
                        'pred_lbl': class_names[preds[idx].item()],
                        'true_lbl': class_names[l_kanji[idx].item()],
                        'conf': conf[idx].item()
                    })

    # Displaying mistakes
    if len(mistakes) > 0:
        fig, axes = plt.subplots(ROWS, COLUMNS, figsize=(15, 12))
        fig.suptitle('Error Gallery (Actual -> Predicted [Confidence])', fontsize=16)
        axes = axes.flatten()
        
        for i, ax in enumerate(axes):
            if i < len(mistakes):
                m = mistakes[i]
                ax.imshow(m['img'])
                
                color = 'red' if m['conf'] > 0.8 else 'orange'
                ax.set_title(f"A: {m['true_lbl']}\nP: {m['pred_lbl']}\nConf: {m['conf']:.2f}", 
                             color=color, fontsize=10, fontweight='bold')
            ax.axis('off')
        plt.tight_layout()
        plt.show()
    else:
        print("No errors found in the visualized sample.")

    # Stats
    df = pd.DataFrame({
        'Actual': [class_names[i] for i in all_labels],
        'Predicted': [class_names[i] for i in all_preds]
    })
    
    errors_df = df[df['Actual'] != df['Predicted']]
    
    if not errors_df.empty:
        print("\n--- TOP KANJIS WITH MOST ERRORS ---")
        print(errors_df['Actual'].value_counts().head(10))
        
        print("\n--- MOST COMMON CONFUSIONS (Actual -> Predicted) ---")
        print(errors_df.groupby(['Actual', 'Predicted']).size().sort_values(ascending=False).head(10))
    
    acc = np.mean(np.array(all_preds) == np.array(all_labels))
    all_probs = np.array(all_probs)
    top3_acc = top_k_accuracy_score(all_labels, all_probs, k=3)
    top5_acc = top_k_accuracy_score(all_labels, all_probs, k=5)
    print(f"\nGlobal Test Accuracy: {acc*100:.2f}%")
    print(f"Top-3 Accuracy: {top3_acc*100:.2f}%")
    print(f"Top-5 Accuracy: {top5_acc*100:.2f}%")

# Execution
try:
    with open(CLASSES_SAVE_PATH, 'r') as f:
        class_names = json.load(f)
    print(f"Classes loaded from {CLASSES_SAVE_PATH} ({len(class_names)} classes).")
except FileNotFoundError:
    print(f"File not found {CLASSES_SAVE_PATH}. Using global variable 'full_dataset'...")
    class_names = full_dataset.classes

model = model.to(device)
visualize_errors_and_metrics(model, test_loader, class_names)

# Color histogram of Test and Inference images

In [ ]:
import matplotlib.pyplot as plt
import torchvision.transforms as T

inputs, _ = next(iter(train_loader))
inputs_val, _ = next(iter(val_loader))
img_train = inputs[0].cpu().numpy().squeeze()
img_val = inputs_val[0].cpu().numpy().squeeze()

# Get inference image 
img_inf = ip.preprocess_image("../TESTS/体.jpeg", IMG_SIZE, 1)
img_inf_tensor = T.ToTensor()(img_inf).numpy().squeeze()

# Comparing histograms
plt.figure(figsize=(10, 4))
plt.subplot(1, 3, 1)
plt.hist(img_train.ravel(), bins=50)
plt.title("Input of model (train)")
plt.subplot(1, 3, 2)
plt.hist(img_val.ravel(), bins=50)
plt.title("Input of model (val-test)")
plt.subplot(1, 3, 3)
plt.hist(img_inf_tensor.ravel(), bins=50)
plt.title("Preprocessed image for inference")
plt.show()


# Loading model for external evaluation

In [ ]:
import torch
from modules.fonts import load_font
import json
from modules.models import build_multi_head_model
from modules.utils import get_device

# Load kanji classes
try:
    with open(CLASSES_SAVE_PATH, 'r') as f:
        class_names = json.load(f)
    print(f"Classes loaded from {CLASSES_SAVE_PATH} ({len(class_names)} classes).")
except FileNotFoundError:
    print(f"File not found {CLASSES_SAVE_PATH}. Using global variable 'full_dataset'...")
    class_names = full_dataset.classes

# Load components mapping
try:
    with open(COMPONENTS_SAVE_PATH, 'r') as f:
        comp_to_idx = json.load(f)
    print(f"Components loaded from {COMPONENTS_SAVE_PATH} ({len(comp_to_idx)} components).")
except FileNotFoundError:
    print(f"File not found {COMPONENTS_SAVE_PATH}. Using global variable 'full_dataset'...")
    comp_to_idx = full_dataset.comp_to_idx

load_font(FONT_PATH)

# Build multi-head model with both kanji and component classes
model = build_multi_head_model(
    num_classes=len(class_names),
    components_classes=len(comp_to_idx)
)

device = get_device()
model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=device))
model = model.to(device)
print(f"Model loaded successfully: {len(class_names)} kanji classes, {len(comp_to_idx)} component classes")

# Evaluation with samples that aren't in the dataset

In [ ]:
from modules.evaluation import predict_and_evaluate

predict_and_evaluate(
    model=model,
    folder_path=PREDICTION_FOLDER,
    class_names=class_names,
    device=device,
    img_size=IMG_SIZE,
    num_channels=CHANEL_SIZE,
    skip_not_in_classes=False,
    display=True
)

# Evaluation with other dataset (CASIA training set)

In [ ]:
print("====== Evaluation with CASIA train set ======")
predict_and_evaluate(
    model=model,
    folder_path=CASIA_DATASET_TRAIN_PATH,
    class_names=class_names,
    device=device,
    img_size=IMG_SIZE,
    num_channels=CHANEL_SIZE,
    skip_not_in_classes=True,
    display=False
)
print("====== Evaluation with CASIA test set ======")
predict_and_evaluate(
    model=model,
    folder_path=CASIA_DATASET_TEST_PATH,
    class_names=class_names,
    device=device,
    img_size=IMG_SIZE,
    num_channels=CHANEL_SIZE,
    skip_not_in_classes=True,
    display=False
)

# Evaluation with samples that aren't in the dataset MC Dropout

In [ ]:
predict_and_evaluate(
    model=model,
    folder_path=PREDICTION_FOLDER,
    class_names=class_names,
    device=device,
    img_size=IMG_SIZE,
    skip_not_in_classes=False,
    display=True,
    num_channels=CHANEL_SIZE,
    mc_iterations=10
)

# Evaluation with other dataset (CASIA training set) MC Dropout

In [ ]:
print("====== Evaluation with CASIA train set (MC) ======")
predict_and_evaluate(
    model=model,
    folder_path=CASIA_DATASET_TRAIN_PATH,
    class_names=class_names,
    device=device,
    img_size=IMG_SIZE,
    skip_not_in_classes=True,
    display=False,
    num_channels=CHANEL_SIZE,
    mc_iterations=10
)
print("====== Evaluation with CASIA test set (MC) ======")
predict_and_evaluate(
    model=model,
    folder_path=CASIA_DATASET_TEST_PATH,
    class_names=class_names,
    device=device,
    img_size=IMG_SIZE,
    skip_not_in_classes=True,
    display=False,
    num_channels=CHANEL_SIZE,
    mc_iterations=10
)